# `@lru_cache` in Python - From Basics to Expert

## What is it?
- `@lru_cache` is a decorator from Python's `functools` module that **memoizes** a function - it remembers the result of previous cals and returns the cached result when the same inputs are seen again, instead of recomputing.
- LRU stands for Least Recently used - when the cache fills up, it evicts the result that accessed least recently to make room for new ones

In [ ]:
from functools import lru_cache

## Why use it?
- speed - avoid repeating expensive computations
- simplicity - one decorator line - no manual dict caching
- correctness - only works on pure functions (same input - same output, no side effects), which is a healthy constraint

The core idea is trading memory for speed.

## When to use it?
#### Good candidates:
- recursive funcitons with overlapping subproblems (Fibonacci, tree traversal)
- pure functions called repeatedly with the same arguments
- expnsive computations: parsing, math, string processing
- database style ookups where data doesn't change

#### Bad candidates:
- functions with side effects (writing to files, printing, mutating state)
- funcitons where the output changes over time (current time, random numbers, live API calls)
- functions with unhashable arguments (lists, dicts) - these will raise a `TypeError`

## The Baseline: Without Cache
Before seeing the cache, understand the problem it solves

In [ ]:
import time

def slow_square(n):
    time.sleep(0.5) # simulate expensive work
    return n * n

# called 4 times, pays the cost every time
start = time.time()
results = [slow_square(5), slow_square(5), slow_square(5), slow_square(5)]

print(f"Took {time.time() - start:.2f}") #~ 2s
print(results)

# Same input, same output, computed four times - WASTEFUL

## Basic usage

In [ ]:
@lru_cache(maxsize=128)
def slow_square(n):
    time.sleep(0.5)
    return n * n

start = time.time()
results = [slow_square(5), slow_square(5), slow_square(5), slow_square(5)]

print(f"Took {time.time() - start:.2f}") #~ 0.5s - only computed once
print(results)

- the efirtst call computes ans stores the result - the next three calls return instantly from cache
- `maxsize` controls hoe many unique results to store - when the cache is full - the east recently used enrty is dropped

## The Classic example - Fibonacci
- Fubonacci is the poster child for `lru_cache` because naive recursion recomputes the same values exponentially

In [ ]:
# WITHOUT CACHE - exponential time O(2^n)

def fib_slow(n):
    if n < 2:
        return n
    return fib_slow(n - 1) + fib_slow(n - 2)

@lru_cache(maxsize=None)
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

start = time.time()
print(fib_slow(25))
print(f"Without cache: {time.time() - start:.3f}s")

start = time.time()
print(fib(25))
print(f"with cache: {time.time() - start:.6f}s")

- `maxsize=None` means unlimited cache - useful when you know the input space is bounded (like `fib(n)` for a fixed max n)

## Inspecting the Cache
- `lru_cache` gives you `.cache_info()` method to peek inside

In [ ]:
@lru_cache(maxsize=4)
def compute(n):
    return n ** 3

# warm up the cache
for i in range(6):
    compute(i)

print(compute.cache_info())


In [ ]:
# already cached
compute(5)
compute(5)

print(compute.cache_info())

- hits - how many times a cached results was returned
- misses - how many times the funciton actually ran
- maxsize - the limit you set
- currsize - how many entries are currently stored

- A high hit/mixx ration means the cache is working well
- a low ration means you are caching things that are rearly reused - not worth the memeory

## Cleating the Cache

In [ ]:
@lru_cache(maxsize=128)
def get_data(key):
    print(f" computing for {key}...")
    return key.upper()

get_data("hello") # computing for "hello"
get_data("hello") # from cache - no print
get_data("world") # computing for world

print(get_data.cache_info())

In [ ]:
get_data.cache_clear()
print(get_data.cache_info())
get_data("hello") # computing for hello... cache was cleared

# use `.cache_clear()` when underlying data changes and you need fresh results

## `maxsize=None` VS `maxsize=128` VS `maxsize=0`

In [ ]:
@lru_cache(maxsize=None) #unlimited cache - frows forever
def func1(n): return n * 2

@lru_cache(maxsize=128) # fixed cache - LRU evitction when full (default)
def func2(n): return n * 2

@lru_cache(maxsize=0) # cache disabled entirely - useful for testing or toggling
def func3(n): return n * 2

# shorthand for maxsize=128 (python 3.8+)
@lru_cache
def func4(n): return n * 2

### Rule of thumb
- `maxsize=None` when inputs are bounded (recursion, small fixed domains)
- `maxsize=128` (or custom) for oprn-ended inputs - cap memeory usage
- `maxsize=0` to disable caching (useful in tests to isolate behavior)

## Arguments must be hashable
- the cache works by using the function arguments as dictionary keys - this means arguments must be hashable

In [ ]:
@lru_cache(maxsize=128)
def process(data):
    print(f" data tyoe {type(data)}")
    return sum(data)

In [ ]:
# works fine - tuples are hashable
print(process((1,2,3))) # 6
print(process((1,2,3))) # 6 from cache

In [ ]:
# raises TypeError - lists are NOThashable
try:
    process([1 ,2, 3])
except TypeError as e:
    print(f"TypeError: {e}")

In [ ]:
# THE FIX: convert mutable types to immutable ones before passing in

@lru_cache(maxsize=128)
def process(data_tuple):
    return sum(data_tuple)

my_list = [1, 2, 3]
result = process(tuple(my_list)) # convert at the call site
print(result)

# Multiple Arguments
Each unique compibation of arguments is cached separately.

In [ ]:
@lru_cache(maxsize=256)
def power(base, exponent):
    print(f" computing {base}^{exponent}")
    return base ** exponent

print(power(2, 10)) # computing 2^10
print(power(2, 10)) # (from cache)
print(power(3, 10)) # computing 3^10
print(power(2, 5)) # computing 2^5
print(power(2, 10)) # (from cache again)

print(power.cache_info())

> Be careful: `power(2, 10)` and `power(10, 2)` are different cache keys even if your math happens to be symetric - the cache sees arguments,  not semantics

# Using with Methods on Classes

- `@lru_cache` on instance methods is tricky becasue `self` becomes part of the cache key, which holds a reference to the instance and preevnts garbage collection

In [ ]:
# DANGEROUS - memory leak
class MyClass:
    @lru_cache(maxsize=128) # self is cached - instance never collected
    def compute(self, n):
        return n ** 2

## SAFE APPROACH 1: use `@staticmethod` or `@classmethod` instead

In [ ]:
class MathHelper:
    @staticmethod
    @lru_cache(maxsize=128)
    def square(n):
        return n * n
    
print(MathHelper.square(7)) # 49
print(MathHelper.square(7)) # (from cache)

print(MathHelper.square.cache_info())

## SAFE APPROACH 2: use `@cached_property` for per-instance, compute-once values (Python 3.8+)

In [ ]:
from functools import cached_property
import math

class Circle:
    def __init__(self, radius):
        self.radius = radius

    @cached_property
    def area(self):
        print("computing area...")
        return math.pi * self.radius ** 2
    
c = Circle(5)
print(c.area) # computing area...
print(c.area) # (cached on instance, no recompute)


# `functools.cache` - The Clean Alias (Python 3.9+)
Python 3.9 introduced `functools.cache` as a cleaner alias for `@lru_cache(maxzise=None)`

In [ ]:
from functools import cache

@cache
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n -2)

print(fib(50))
print(fib.cache_info())

# Real-world Pattern - Caching Config/File Reads

In [ ]:
import json

@lru_cache(maxsize=128)
def load_config(filepath: str) -> dict:
    """Loads a JSON config file - cached so disk is only read once per path"""
    print(f"reading from disk: {filepath}")
    with open(filepath) as file:
        return json.load(file)
    
load_config('assets/mock_domain_settings.json') # reads disk
load_config('assets/mock_domain_settings.json') # retrun cached result  
load_config('assets/mock_infra_settings.json') # reads disk (different key)

# this is a common patttern: functions that read static data (config, lookup tables, compiled regexes) benefit hugely from being cached

# Expert Pattern - Cache Busting with Version Key
When you need control over when to invalidate without calling `cache_clear()` globally

In [ ]:
@lru_cache(maxsize=128)
def get_user_data(user_id: int, _version: int = 0):
    """_version is a cache-busting key - increment it to force recomputation"""
    print(f"fetching user {user_id} (version {_version})")
    return {
        "id": user_id,
        "name": f" User {user_id}"
    }

# normal usage - verison 0 by default
get_user_data(42) # fetches
get_user_data(42) # cached

# forces recompute for user 42 only
get_user_data(42, _version=1) # fetches again (new key)
get_user_data(42, _version=1) # cached under new version

# Benchmarking the Cache

In [ ]:
import time

def benchmark(func, args_list, label):
    func.cache_clear()
    start = time.perf_counter()

    for args in args_list:
        func(*args)

    elapsed = time.perf_counter() - start
    info = func.cache_info()

    print(f"{label}: {elapsed: .4f}s | {info}")

@lru_cache(maxsize=64)
def expensive(n):
    total = 0
    for i in range(n * 1000):
        total += 1
    return total

# lots of repteated calls
repeated_args = [(i % 20,) for i in range(200)]

benchmark(expensive, repeated_args, "with LRU cache")

# compare without cache benefit (all unique inputs)
unique_args = [(i,) for i in range(200)]
benchmark(expensive, unique_args, "all cache misses")

# Cheat Sheet

| Scenario                                        | Recommendation                        |
| --                                              | --                                    |
| recursive function with overlapping subproblems | `@lru_cache(maxsize=None)` or `cache` |
| repeated calls with a bounded input domain      | `@lru_cache(maxsize=N)`               |
| instance method                                 | `@staticmethod` + `@lru_cache` or `@cached_property` |
| argument is a list/dics                         | convert to `tuple`/`fronzenset` first |
| need to invalidate for specific inputs          | cache-busting version key |
| need to invalidate everything                   | `.cache_clear()` |
| want to inspect performance                     | `.cache_info() |
| python 3.9 and want simples syntax | `@cache` |

In [ ]:
# Common cotchas summary

# 1. do not cache functions with side ffects
@lru_cache(maxsize=128)
def bad_logger(n):
    print(f" processing {n}") #only prints once - then cached silently
    return n * 2


# 2. do not forget unhashable args raise TypeError
# @lru_cache won't work with list/dict arguments

# 3. maxsize=None on a function with infinite input domain = memory leak
# always set a maxsize if inputs are unboundeed

# 4. caching a method on self holds a strong ref - prevents Garbage Collector
# use @staticmethod or @cached_property instead

# 5. cache persists across the program's lifetime by default
# call .cache_clear() between tests to avoid state bleed